# 01 — Scrape Guardian News API
Fetches articles tagged with AI-related keywords from The Guardian API.
Output: `data/raw_articles.csv`

In [2]:
import requests
import pandas as pd
import time
import os

os.makedirs('data', exist_ok=True)

API_KEY = 'ab4de98f-45b1-42d8-b85f-df65936132e3'
BASE_URL = 'https://content.guardianapis.com/search'

# Query params — adjust as needed
QUERY = 'artificial intelligence OR machine learning OR generative AI OR AI regulation OR AI safety'
FROM_DATE = '2022-01-01'
TO_DATE   = '2025-12-31'
PAGE_SIZE = 200  # max per request


In [3]:
def fetch_page(page: int) -> dict:
    params = {
        'q': QUERY,
        'from-date': FROM_DATE,
        'to-date': TO_DATE,
        'page-size': PAGE_SIZE,
        'page': page,
        'show-fields': 'bodyText,headline,wordcount',
        'api-key': API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    r.raise_for_status()
    return r.json()['response']


def parse_results(results: list) -> list:
    rows = []
    for item in results:
        fields = item.get('fields', {})
        rows.append({
            'id':        item.get('id'),
            'date':      item.get('webPublicationDate', '')[:10],  # YYYY-MM-DD
            'section':   item.get('sectionName'),
            'headline':  fields.get('headline', ''),
            'body':      fields.get('bodyText', ''),
            'wordcount': fields.get('wordcount', 0),
            'url':       item.get('webUrl')
        })
    return rows


In [4]:
# Paginate through all results
all_rows = []
page = 1

first = fetch_page(1)
total_pages = first['pages']
print(f'Total articles: {first["total"]} | Pages: {total_pages}')

all_rows.extend(parse_results(first['results']))

for page in range(2, total_pages + 1):
    resp = fetch_page(page)
    all_rows.extend(parse_results(resp['results']))
    print(f'  Page {page}/{total_pages} — total so far: {len(all_rows)}')
    time.sleep(0.5)  # stay within rate limit

Total articles: 24291 | Pages: 122
  Page 2/122 — total so far: 400
  Page 3/122 — total so far: 600
  Page 4/122 — total so far: 800
  Page 5/122 — total so far: 1000
  Page 6/122 — total so far: 1200
  Page 7/122 — total so far: 1400
  Page 8/122 — total so far: 1600
  Page 9/122 — total so far: 1800
  Page 10/122 — total so far: 2000
  Page 11/122 — total so far: 2200
  Page 12/122 — total so far: 2400
  Page 13/122 — total so far: 2600
  Page 14/122 — total so far: 2800
  Page 15/122 — total so far: 3000
  Page 16/122 — total so far: 3200
  Page 17/122 — total so far: 3400
  Page 18/122 — total so far: 3600
  Page 19/122 — total so far: 3800
  Page 20/122 — total so far: 4000
  Page 21/122 — total so far: 4200
  Page 22/122 — total so far: 4400
  Page 23/122 — total so far: 4600
  Page 24/122 — total so far: 4800
  Page 25/122 — total so far: 5000
  Page 26/122 — total so far: 5200
  Page 27/122 — total so far: 5400
  Page 28/122 — total so far: 5600
  Page 29/122 — total so far: 5

HTTPError: 503 Server Error: Service Unavailable for url: https://content.guardianapis.com/search?q=artificial+intelligence+OR+machine+learning+OR+generative+AI+OR+AI+regulation+OR+AI+safety&from-date=2022-01-01&to-date=2025-12-31&page-size=200&page=121&show-fields=bodyText%2Cheadline%2Cwordcount&api-key=ab4de98f-45b1-42d8-b85f-df65936132e3

In [5]:
df = pd.DataFrame(all_rows)
df['date'] = pd.to_datetime(df['date'])
df.drop_duplicates(subset='id', inplace=True)

print(f'\nFinal dataset: {len(df)} articles')
df.head()


Final dataset: 23912 articles


,id,date,section,headline,body,wordcount,url
0,us-news/2025/nov/13/rejecting-generative-ai-in...,2025-11-13,US news,Rejecting generative AI in healthcare won’t pr...,Eric Reinhart’s essay raises thoughtful concer...,322,https://www.theguardian.com/us-news/2025/nov/1...
1,games/2025/nov/19/pushing-buttons-arc-raiders-...,2025-11-19,Games,How generative AI in Arc Raiders started a scr...,"Arc Raiders is, by all accounts, a late game-o...",1602,https://www.theguardian.com/games/2025/nov/19/...
2,technology/2025/jun/03/honest-ai-yoshua-bengio,2025-06-03,Technology,AI pioneer announces non-profit to develop ‘ho...,An artificial intelligence pioneer has launche...,593,https://www.theguardian.com/technology/2025/ju...
3,us-news/2025/dec/11/trump-executive-order-arti...,2025-12-12,US news,Trump signs executive order aimed at preventin...,Donald Trump signed an executive order on Thur...,821,https://www.theguardian.com/us-news/2025/dec/1...
4,technology/2025/sep/16/how-ai-is-undermining-l...,2025-09-16,Technology,How AI is undermining learning and teaching in...,In discussing generative artificial intelligen...,259,https://www.theguardian.com/technology/2025/se...


In [6]:
# Basic quality filter
df = df[df['wordcount'].astype(int) >= 100]  # drop very short pieces
df = df[df['body'].str.strip() != '']

df.to_csv('data/raw_articles.csv', index=False)
print('Saved: data/raw_articles.csv')
df.info()

Saved: data/raw_articles.csv
<class 'pandas.DataFrame'>
Index: 23859 entries, 0 to 23999
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   id         23859 non-null  str           
 1   date       23859 non-null  datetime64[us]
 2   section    23859 non-null  str           
 3   headline   23859 non-null  str           
 4   body       23859 non-null  str           
 5   wordcount  23859 non-null  str           
 6   url        23859 non-null  str           
dtypes: datetime64[us](1), str(6)
memory usage: 1.5 MB
